In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score

FEATURE_PATH = "../results/training_features.csv"

df = pd.read_csv(FEATURE_PATH)

FEATURE_COLUMNS = [
    "name_exact",
    "name_token_jaccard",
    "name_char_similarity",
    "name_token_set_ratio",
    "name_token_sort_ratio",
    "name_shared_tokens",
    "name_length_ratio",
    "address_exact",
    "address_token_jaccard",
    "address_char_similarity",
    "address_token_set_ratio",
    "address_token_sort_ratio",
    "address_shared_tokens",
    "address_length_ratio",
    "country_match",
    "name_missing",
    "address_missing",
]

X = df[FEATURE_COLUMNS]
y = df["label"]
groups = df["source1_entity_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=groups)
)

X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

val_meta = df.iloc[val_idx][
    ["source1_entity_id", "label"]
].copy()

print("Validation pairs:", len(X_val))
print("Validation S1:", val_meta["source1_entity_id"].nunique())
print("Validation positives:", y_val.sum())

Validation pairs: 248185
Validation S1: 1995
Validation positives: 6403


In [2]:
import xgboost as xgb

XGB_MODEL_PATH = "../results/xgboost_entity_resolution.json"

xgb_model = xgb.XGBClassifier()
xgb_model.load_model(XGB_MODEL_PATH)

print("XGBoost JSON model loaded")

XGBoost JSON model loaded


In [5]:
# Create validation prediction dataframe

val_pred = val_meta.copy()

val_pred["probability"] = xgb_val_prob

print("Validation predictions:", len(val_pred))
print(val_pred.head())

Validation predictions: 248185
     source1_entity_id  label   probability
1373      S1-680786781      0  7.074547e-07
1374      S1-680786781      0  2.169920e-05
1375      S1-680786781      0  1.251994e-05
1376      S1-680786781      0  6.786254e-05
1377      S1-680786781      0  2.339064e-06


In [6]:
BASE_THRESHOLD = 0.67

val_pred["baseline_pred"] = (
    val_pred["probability"] >= BASE_THRESHOLD
).astype(int)

precision = precision_score(
    val_pred["label"],
    val_pred["baseline_pred"],
    zero_division=0
)

recall = recall_score(
    val_pred["label"],
    val_pred["baseline_pred"],
    zero_division=0
)

f1 = f1_score(
    val_pred["label"],
    val_pred["baseline_pred"],
    zero_division=0
)

f05 = fbeta_score(
    val_pred["label"],
    val_pred["baseline_pred"],
    beta=0.5,
    zero_division=0
)

print("Baseline XGBoost")
print("Threshold:", BASE_THRESHOLD)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("F0.5:", f05)

Baseline XGBoost
Threshold: 0.67
Precision: 0.9710869177226485
Recall: 0.8497579259722006
F1: 0.9063801432617025
F0.5: 0.94412632309561


In [8]:
X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()

print("Train pairs:", len(X_train))
print("Validation pairs:", len(X_val))
print("Train positives:", y_train.sum())
print("Validation positives:", y_val.sum())

Train pairs: 1011664
Validation pairs: 248185
Train positives: 25309
Validation positives: 6403


In [9]:
import xgboost as xgb

xgb_val_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_val_model.fit(
    X_train,
    y_train
)

print("Validation-only XGBoost trained.")

Validation-only XGBoost trained.


In [10]:
xgb_val_prob = xgb_val_model.predict_proba(X_val)[:, 1]

val_pred["probability"] = xgb_val_prob

print("Validation probabilities:", len(xgb_val_prob))

Validation probabilities: 248185


In [11]:
BASE_THRESHOLD = 0.67

val_pred["baseline_pred"] = (
    val_pred["probability"] >= BASE_THRESHOLD
).astype(int)

precision = precision_score(
    val_pred["label"],
    val_pred["baseline_pred"],
    zero_division=0
)

recall = recall_score(
    val_pred["label"],
    val_pred["baseline_pred"],
    zero_division=0
)

f1 = f1_score(
    val_pred["label"],
    val_pred["baseline_pred"],
    zero_division=0
)

f05 = fbeta_score(
    val_pred["label"],
    val_pred["baseline_pred"],
    beta=0.5,
    zero_division=0
)

print("Validation-only XGBoost")
print("Threshold:", BASE_THRESHOLD)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("F0.5:", f05)

Validation-only XGBoost
Threshold: 0.67
Precision: 0.9555394903307428
Recall: 0.8257066999843823
F1: 0.8858914209115282
F0.5: 0.9264061678640266


In [12]:
# S1-relative probability analysis

val_pred["max_probability"] = (
    val_pred.groupby("source1_entity_id")["probability"]
    .transform("max")
)

val_pred["relative_probability"] = (
    val_pred["probability"] / val_pred["max_probability"]
)

print(val_pred[
    [
        "source1_entity_id",
        "probability",
        "max_probability",
        "relative_probability",
        "label"
    ]
].head(20))

     source1_entity_id   probability  max_probability  relative_probability  \
1373      S1-680786781  8.207548e-07         0.998409          8.220626e-07   
1374      S1-680786781  2.811449e-05         0.998409          2.815928e-05   
1375      S1-680786781  9.239070e-06         0.998409          9.253791e-06   
1376      S1-680786781  3.095397e-05         0.998409          3.100329e-05   
1377      S1-680786781  1.815919e-06         0.998409          1.818813e-06   
1378      S1-680786781  1.316950e-05         0.998409          1.319049e-05   
1379      S1-680786781  3.361622e-06         0.998409          3.366978e-06   
1380      S1-680786781  1.027668e-05         0.998409          1.029305e-05   
1381      S1-680786781  1.227494e-05         0.998409          1.229450e-05   
1382      S1-680786781  6.081519e-07         0.998409          6.091209e-07   
1383      S1-680786781  1.338161e-06         0.998409          1.340293e-06   
1384      S1-680786781  3.879517e-06         0.99840

In [13]:
results = []

for relative_threshold in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:

    pred = (
        (val_pred["probability"] >= 0.67) &
        (val_pred["relative_probability"] >= relative_threshold)
    ).astype(int)

    precision = precision_score(
        val_pred["label"],
        pred,
        zero_division=0
    )

    recall = recall_score(
        val_pred["label"],
        pred,
        zero_division=0
    )

    f1 = f1_score(
        val_pred["label"],
        pred,
        zero_division=0
    )

    f05 = fbeta_score(
        val_pred["label"],
        pred,
        beta=0.5,
        zero_division=0
    )

    results.append({
        "relative_threshold": relative_threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f0.5": f05
    })

relative_results = pd.DataFrame(results).sort_values(
    "f0.5",
    ascending=False
)

relative_results

,relative_threshold,precision,recall,f1,f0.5
3,0.75,0.967145,0.799938,0.875630,0.928336
2,0.70,0.959729,0.818835,0.883701,0.927800
1,0.60,0.955539,0.825707,0.885891,0.926406
0,0.50,0.955539,0.825707,0.885891,0.926406
4,0.80,0.971339,0.778073,0.864031,0.925369
5,0.85,0.977601,0.749805,0.848683,0.921603
6,0.90,0.983054,0.715758,0.828378,0.914734
7,0.95,0.987909,0.650789,0.784672,0.895166


In [14]:
fine_results = []

for relative_threshold in np.arange(0.68, 0.811, 0.01):

    pred = (
        (val_pred["probability"] >= 0.67) &
        (val_pred["relative_probability"] >= relative_threshold)
    ).astype(int)

    precision = precision_score(
        val_pred["label"],
        pred,
        zero_division=0
    )

    recall = recall_score(
        val_pred["label"],
        pred,
        zero_division=0
    )

    f1 = f1_score(
        val_pred["label"],
        pred,
        zero_division=0
    )

    f05 = fbeta_score(
        val_pred["label"],
        pred,
        beta=0.5,
        zero_division=0
    )

    fine_results.append({
        "relative_threshold": round(relative_threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f0.5": f05
    })

fine_results = pd.DataFrame(fine_results).sort_values(
    "f0.5",
    ascending=False
)

fine_results.head(10)

,relative_threshold,precision,recall,f1,f0.5
5,0.73,0.964585,0.808215,0.879504,0.928651
6,0.74,0.966029,0.803842,0.877504,0.928559
4,0.72,0.962949,0.811807,0.880942,0.928380
7,0.75,0.967145,0.799938,0.875630,0.928336
8,0.76,0.967589,0.797283,0.874219,0.927946
2,0.70,0.959729,0.818835,0.883701,0.927800
9,0.77,0.969060,0.792441,0.871896,0.927707
3,0.71,0.960943,0.814618,0.881751,0.927619
1,0.69,0.957931,0.821490,0.884480,0.927134
10,0.78,0.970164,0.787131,0.869115,0.927050


In [15]:
BEST_RELATIVE = 0.73

baseline_mask = val_pred["probability"] >= 0.67

relative_mask = (
    baseline_mask &
    (val_pred["relative_probability"] >= BEST_RELATIVE)
)

print("Baseline predicted matches:", baseline_mask.sum())
print("Relative predicted matches:", relative_mask.sum())
print("Removed:", baseline_mask.sum() - relative_mask.sum())
print(
    "Removal rate:",
    (baseline_mask.sum() - relative_mask.sum()) / baseline_mask.sum()
)

Baseline predicted matches: 5533
Relative predicted matches: 5365
Removed: 168
Removal rate: 0.030363274896078075


In [16]:
removed_mask = baseline_mask & ~relative_mask

removed = val_pred.loc[
    removed_mask,
    ["source1_entity_id", "probability", "max_probability",
     "relative_probability", "label"]
].copy()

print("Removed pairs:", len(removed))
print("True positives removed:", removed["label"].sum())
print("False positives removed:", (removed["label"] == 0).sum())

print("\nRemoved-pair precision:")
print(removed["label"].mean())

Removed pairs: 168
True positives removed: 112
False positives removed: 56

Removed-pair precision:
0.6666666666666666


In [17]:
print("\nAll baseline predicted pairs:")
baseline_pred_df = val_pred.loc[
    baseline_mask,
    ["label"]
]

print("TP:", baseline_pred_df["label"].sum())
print("FP:", (baseline_pred_df["label"] == 0).sum())

print("\nRemoved probability statistics:")
print(
    removed[
        ["probability", "max_probability", "relative_probability"]
    ].describe()
)


All baseline predicted pairs:
TP: 5287
FP: 246

Removed probability statistics:
       probability  max_probability  relative_probability
count   168.000000       168.000000            168.000000
mean      0.698780         0.993441              0.703422
std       0.016651         0.010493              0.016294
min       0.670104         0.940924              0.671435
25%       0.684540         0.992517              0.690293
50%       0.698877         0.998190              0.704533
75%       0.712933         0.999504              0.718348
max       0.728150         0.999888              0.729754


In [18]:
# XGBoost + selective LightGBM borderline rule

XGB_THRESHOLD = 0.67
LGB_BORDERLINE = 0.73

# Start with XGBoost decision
hybrid_pred = (val_pred["probability"] >= XGB_THRESHOLD).astype(int)

# Borderline region: 0.67 <= XGB < 0.73
borderline_mask = (
    (val_pred["probability"] >= XGB_THRESHOLD) &
    (val_pred["probability"] < LGB_BORDERLINE)
)

# Let LightGBM decide ONLY for borderline pairs
hybrid_pred[borderline_mask] = (
    val_pred.loc[borderline_mask, "relative_probability"] >= LGB_BORDERLINE
).astype(int)

# Metrics
precision = precision_score(
    val_pred["label"],
    hybrid_pred,
    zero_division=0
)

recall = recall_score(
    val_pred["label"],
    hybrid_pred,
    zero_division=0
)

f1 = f1_score(
    val_pred["label"],
    hybrid_pred,
    zero_division=0
)

f05 = fbeta_score(
    val_pred["label"],
    hybrid_pred,
    beta=0.5,
    zero_division=0
)

print("XGBoost + Selective LightGBM")
print("XGB threshold:", XGB_THRESHOLD)
print("LightGBM threshold:", LGB_BORDERLINE)
print("Borderline pairs:", borderline_mask.sum())
print("Predicted matches:", hybrid_pred.sum())
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("F0.5:", f05)

XGBoost + Selective LightGBM
XGB threshold: 0.67
LightGBM threshold: 0.73
Borderline pairs: 242
Predicted matches: 5365
Precision: 0.9645852749301025
Recall: 0.8082148992659691
F1: 0.8795037389530931
F0.5: 0.9286508990417399


In [19]:
for lgb_threshold in [0.72, 0.73, 0.74, 0.75]:
    pred = (val_pred["probability"] >= 0.67).astype(int)

    borderline = (
        (val_pred["probability"] >= 0.67) &
        (val_pred["probability"] < lgb_threshold)
    )

    pred[borderline] = (
        val_pred.loc[borderline, "relative_probability"] >= lgb_threshold
    ).astype(int)

    p = precision_score(val_pred["label"], pred, zero_division=0)
    r = recall_score(val_pred["label"], pred, zero_division=0)
    f1 = f1_score(val_pred["label"], pred, zero_division=0)
    f05 = fbeta_score(
        val_pred["label"], pred,
        beta=0.5,
        zero_division=0
    )

    print(
        f"LGB threshold={lgb_threshold:.2f} | "
        f"Borderline={borderline.sum()} | "
        f"Precision={p:.6f} | "
        f"Recall={r:.6f} | "
        f"F1={f1:.6f} | "
        f"F0.5={f05:.6f}"
    )

LGB threshold=0.72 | Borderline=197 | Precision=0.962949 | Recall=0.811807 | F1=0.880942 | F0.5=0.928380
LGB threshold=0.73 | Borderline=242 | Precision=0.964585 | Recall=0.808215 | F1=0.879504 | F0.5=0.928651
LGB threshold=0.74 | Borderline=283 | Precision=0.966029 | Recall=0.803842 | F1=0.877504 | F0.5=0.928559
LGB threshold=0.75 | Borderline=316 | Precision=0.967145 | Recall=0.799938 | F1=0.875630 | F0.5=0.928336


In [20]:
print("XGB results columns:")
print(xgb_results.columns.tolist())

print("\nLightGBM reranked columns:")
print(lgb_reranked_pairs.columns.tolist())

print("\nXGB rows:", len(xgb_results))
print("LightGBM reranked pairs:", len(lgb_reranked_pairs))

XGB results columns:


NameError: name 'xgb_results' is not defined

In [23]:
import pandas as pd

xgb_sample = pd.read_parquet(
    RESULTS_DIR / "inference_checkpoints" / "chunk_00000.parquet"
)

lgb_sample = pd.read_parquet(
    RESULTS_DIR / "lgb_rerank_checkpoints" / "chunk_00000.parquet"
)

print("XGB checkpoint:")
print(xgb_sample.columns.tolist())
print(xgb_sample.shape)
print(xgb_sample.head())

print("\nLightGBM rerank checkpoint:")
print(lgb_sample.columns.tolist())
print(lgb_sample.shape)
print(lgb_sample.head())

XGB checkpoint:
['source1_entity_id', 'matched_entity_ids']
(2000, 2)
  source1_entity_id                                 matched_entity_ids
0      S1-714132312                          S3-625880872,S3-867809779
1      S1-106407869                          S2-705547832,S3-613056593
2      S1-156285671             S2-172049742,S2-648730048,S3-871416040
3      S1-689823050             S2-212601842,S2-319486263,S2-829077756
4      S1-921369899  S2-808920974,S2-995426671,S3-285097097,S3-5759...

LightGBM rerank checkpoint:
['source1_entity_id', 'candidate_entity_id', 'lgb_probability']
(48346, 3)
  source1_entity_id candidate_entity_id  lgb_probability
0      S1-714132312        S3-625880872         0.998688
1      S1-714132312        S3-867809779         0.976911
2      S1-106407869        S2-705547832         0.960914
3      S1-106407869        S3-613056593         0.973223
4      S1-156285671        S2-172049742         0.999205


In [24]:
from pathlib import Path

for folder_name in ["inference_cache", "inference_checkpoints", "lgb_inference_checkpoints", "lgb_rerank_checkpoints"]:
    folder = RESULTS_DIR / folder_name
    print(f"\n===== {folder_name} =====")
    if folder.exists():
        files = sorted(folder.glob("*"))
        print("Files:", len(files))
        for f in files[:10]:
            print(" ", f.name)


===== inference_cache =====
Files: 2
  test_candidate_features.parquet
  test_s1_features.parquet

===== inference_checkpoints =====
Files: 867
  chunk_00000.parquet
  chunk_00001.parquet
  chunk_00002.parquet
  chunk_00003.parquet
  chunk_00004.parquet
  chunk_00005.parquet
  chunk_00006.parquet
  chunk_00007.parquet
  chunk_00008.parquet
  chunk_00009.parquet

===== lgb_inference_checkpoints =====
Files: 26
  chunk_00000.parquet
  chunk_00001.parquet
  chunk_00002.parquet
  chunk_00003.parquet
  chunk_00004.parquet
  chunk_00005.parquet
  chunk_00006.parquet
  chunk_00007.parquet
  chunk_00008.parquet
  chunk_00009.parquet

===== lgb_rerank_checkpoints =====
Files: 89
  chunk_00000.parquet
  chunk_00001.parquet
  chunk_00002.parquet
  chunk_00003.parquet
  chunk_00004.parquet
  chunk_00005.parquet
  chunk_00006.parquet
  chunk_00007.parquet
  chunk_00008.parquet
  chunk_00009.parquet


In [25]:
import xgboost as xgb
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("../results").resolve()
CACHE_DIR = RESULTS_DIR / "inference_cache"
RERANK_DIR = RESULTS_DIR / "lgb_rerank_checkpoints"

# Load XGBoost JSON
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(RESULTS_DIR / "xgboost_entity_resolution.json")

print("XGBoost JSON loaded.")

# Inspect caches
s1_cache = pd.read_parquet(CACHE_DIR / "test_s1_features.parquet")
candidate_cache = pd.read_parquet(CACHE_DIR / "test_candidate_features.parquet")

print("\nS1 cache:")
print(s1_cache.shape)
print(s1_cache.columns.tolist())

print("\nCandidate cache:")
print(candidate_cache.shape)
print(candidate_cache.columns.tolist())

XGBoost JSON loaded.

S1 cache:
(1732544, 9)
['entity_id', 'business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'country_clean', 'name_missing', 'address_missing']

Candidate cache:
(9969589, 8)
['business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'country_clean', 'name_missing', 'address_missing']


In [3]:
from rapidfuzz import process, fuzz
def build_chunk_features_fast(candidate_pairs_chunk):
    """
    Build the exact same 17 features used during training,
    but calculate the expensive fuzzy features in bulk.
    """

    # --------------------------------------------------
    # 1. Flatten candidate_pairs into individual pairs
    # --------------------------------------------------

    s1_ids = []
    candidate_ids = []

    for row in candidate_pairs_chunk.itertuples(index=False):

        s1_id = row.source1_entity_id
        candidate_string = row.candidate_entity_ids

        if pd.isna(candidate_string) or not candidate_string:
            continue

        ids = [
            x.strip()
            for x in candidate_string.split(",")
            if x.strip()
        ]

        for cid in ids:
            s1_ids.append(s1_id)
            candidate_ids.append(cid)

    if not candidate_ids:
        return (
            np.empty((0, 17), dtype=np.float32),
            [],
            []
        )

    # --------------------------------------------------
    # 2. Batch lookup S1 records
    # --------------------------------------------------

    s1_frame = s1_cache.reindex(s1_ids)

    # --------------------------------------------------
    # 3. Batch lookup candidate records
    # --------------------------------------------------

    candidate_frame = candidate_cache.reindex(
        candidate_ids
    )

    # --------------------------------------------------
    # 4. Convert strings to arrays
    # --------------------------------------------------

    s1_names = (
        s1_frame["norm_name"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    candidate_names = (
        candidate_frame["norm_name"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    s1_addresses = (
        s1_frame["norm_address"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    candidate_addresses = (
        candidate_frame["norm_address"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    s1_countries = (
        s1_frame["country_clean"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    candidate_countries = (
        candidate_frame["country_clean"]
        .fillna("")
        .to_numpy(dtype=object)
    )

    # --------------------------------------------------
    # 5. Exact matches
    # --------------------------------------------------

    name_exact = (
        (s1_names != "")
        & (s1_names == candidate_names)
    ).astype(np.float32)

    address_exact = (
        (s1_addresses != "")
        & (s1_addresses == candidate_addresses)
    ).astype(np.float32)

    country_match = (
        (s1_countries != "")
        & (s1_countries == candidate_countries)
    ).astype(np.float32)

    # --------------------------------------------------
    # 6. Missing flags
    # --------------------------------------------------

    name_missing = np.logical_or(
        s1_frame["name_missing"].to_numpy(dtype=np.int8),
        candidate_frame["name_missing"].to_numpy(dtype=np.int8)
    ).astype(np.float32)
        
    address_missing = np.logical_or(
        s1_frame["address_missing"].to_numpy(dtype=np.int8),
        candidate_frame["address_missing"].to_numpy(dtype=np.int8)
    ).astype(np.float32)

    # --------------------------------------------------
    # 7. String lengths
    # --------------------------------------------------

    s1_name_len = np.array(
        [len(x) for x in s1_names],
        dtype=np.float32
    )

    c_name_len = np.array(
        [len(x) for x in candidate_names],
        dtype=np.float32
    )

    name_length_ratio = np.divide(
        np.minimum(s1_name_len, c_name_len),
        np.maximum(s1_name_len, c_name_len),
        out=np.zeros_like(s1_name_len),
        where=np.maximum(s1_name_len, c_name_len) > 0
    )

    s1_addr_len = np.array(
        [len(x) for x in s1_addresses],
        dtype=np.float32
    )

    c_addr_len = np.array(
        [len(x) for x in candidate_addresses],
        dtype=np.float32
    )

    address_length_ratio = np.divide(
        np.minimum(s1_addr_len, c_addr_len),
        np.maximum(s1_addr_len, c_addr_len),
        out=np.zeros_like(s1_addr_len),
        where=np.maximum(s1_addr_len, c_addr_len) > 0
    )

    # --------------------------------------------------
    # 8. RapidFuzz — BULK / MULTI-CORE
    # --------------------------------------------------

    name_char_similarity = (
        process.cpdist(
            s1_names,
            candidate_names,
            scorer=fuzz.ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    name_token_set_ratio = (
        process.cpdist(
            s1_names,
            candidate_names,
            scorer=fuzz.token_set_ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    name_token_sort_ratio = (
        process.cpdist(
            s1_names,
            candidate_names,
            scorer=fuzz.token_sort_ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    address_char_similarity = (
        process.cpdist(
            s1_addresses,
            candidate_addresses,
            scorer=fuzz.ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    address_token_set_ratio = (
        process.cpdist(
            s1_addresses,
            candidate_addresses,
            scorer=fuzz.token_set_ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    address_token_sort_ratio = (
        process.cpdist(
            s1_addresses,
            candidate_addresses,
            scorer=fuzz.token_sort_ratio,
            workers=-1,
            dtype=np.float32
        ) / 100.0
    )

    # --------------------------------------------------
    # IMPORTANT:
    # Training explicitly sets fuzzy values to 0
    # when either side is empty.
    #
    # RapidFuzz may return 100 for empty vs empty,
    # so correct those rows.
    # --------------------------------------------------

    name_invalid = (
        (s1_names == "")
        | (candidate_names == "")
    )

    address_invalid = (
        (s1_addresses == "")
        | (candidate_addresses == "")
    )

    name_char_similarity[name_invalid] = 0.0
    name_token_set_ratio[name_invalid] = 0.0
    name_token_sort_ratio[name_invalid] = 0.0

    address_char_similarity[address_invalid] = 0.0
    address_token_set_ratio[address_invalid] = 0.0
    address_token_sort_ratio[address_invalid] = 0.0

    # --------------------------------------------------
    # 9. Token overlap features
    #
    # Calculate token sets once per UNIQUE string,
    # rather than once for every candidate pair.
    # --------------------------------------------------

    name_token_cache = {
        text: set(text.split())
        for text in pd.unique(
            np.concatenate([
                s1_names,
                candidate_names
            ])
        )
        if text
    }

    address_token_cache = {
        text: set(text.split())
        for text in pd.unique(
            np.concatenate([
                s1_addresses,
                candidate_addresses
            ])
        )
        if text
    }

    name_shared_tokens = np.zeros(
        len(s1_ids),
        dtype=np.float32
    )

    name_token_jaccard = np.zeros(
        len(s1_ids),
        dtype=np.float32
    )

    address_shared_tokens = np.zeros(
        len(s1_ids),
        dtype=np.float32
    )

    address_token_jaccard = np.zeros(
        len(s1_ids),
        dtype=np.float32
    )

    for i in range(len(s1_ids)):

        # Name
        a = name_token_cache.get(
            s1_names[i],
            set()
        )

        b = name_token_cache.get(
            candidate_names[i],
            set()
        )

        intersection = len(a & b)
        union = len(a | b)

        name_shared_tokens[i] = intersection

        if union:
            name_token_jaccard[i] = (
                intersection / union
            )

        # Address
        a = address_token_cache.get(
            s1_addresses[i],
            set()
        )

        b = address_token_cache.get(
            candidate_addresses[i],
            set()
        )

        intersection = len(a & b)
        union = len(a | b)

        address_shared_tokens[i] = intersection

        if union:
            address_token_jaccard[i] = (
                intersection / union
            )

    # --------------------------------------------------
    # 10. EXACT SAME FEATURE ORDER AS TRAINING
    # --------------------------------------------------

    X = np.column_stack([
        name_exact,
        name_token_jaccard,
        name_char_similarity,
        name_token_set_ratio,
        name_token_sort_ratio,
        name_shared_tokens,
        name_length_ratio,

        address_exact,
        address_token_jaccard,
        address_char_similarity,
        address_token_set_ratio,
        address_token_sort_ratio,
        address_shared_tokens,
        address_length_ratio,

        country_match,
        name_missing,
        address_missing
    ]).astype(np.float32)

    return X, s1_ids, candidate_ids

In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from pathlib import Path

RESULTS_DIR = Path("../results").resolve()
RERANK_DIR = RESULTS_DIR / "lgb_rerank_checkpoints"
OUT_DIR = RESULTS_DIR / "hybrid_xgb_prob_checkpoints"
OUT_DIR.mkdir(exist_ok=True)

# Load model
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(RESULTS_DIR / "xgboost_entity_resolution.json")

XGB_THRESHOLD = 0.67
LGB_THRESHOLD = 0.73

# Get rerank checkpoints
files = sorted(RERANK_DIR.glob("chunk_*.parquet"))

print("Rerank checkpoints:", len(files))

# We need the feature-building function from notebook 08.
print("Checking feature builder...")

if "build_chunk_features_fast" not in globals():
    raise RuntimeError(
        "build_chunk_features_fast is not defined. "
        "Run the optimized feature-builder cell from notebook 08 first."
    )

print("Feature builder found.")

Rerank checkpoints: 89
Checking feature builder...
Feature builder found.


In [8]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("../results").resolve()
CACHE_DIR = RESULTS_DIR / "inference_cache"

s1_cache = pd.read_parquet(
    CACHE_DIR / "test_s1_features.parquet"
)

candidate_cache = pd.read_parquet(
    CACHE_DIR / "test_candidate_features.parquet"
)

print("S1 cache:", s1_cache.shape)
print("Candidate cache:", candidate_cache.shape)

S1 cache: (1732544, 9)
Candidate cache: (9969589, 8)


In [19]:
# Reconstruct XGB probabilities only for the 4.4M reranked pairs

for i, path in enumerate(files):

    out_path = OUT_DIR / path.name

    if out_path.exists():
        print(f"[{i+1}/{len(files)}] Skipping {path.name}")
        continue

    df = pd.read_parquet(path)

    print(
        f"[{i+1}/{len(files)}] {path.name} "
        f"rows={len(df):,}"
    )
    
    # Feature builder expects candidate_entity_ids
    # but rerank checkpoints store candidate_entity_id
    feature_input = df[
        ["source1_entity_id", "candidate_entity_id"]
    ].rename(
        columns={"candidate_entity_id": "candidate_entity_ids"}
    )
    
    X, _, _ = build_chunk_features_fast(
        feature_input
    )

    print("Feature rows:", X.shape[0])
    print("Feature columns:", X.shape[1])
    
    print("Feature matrix:", X.shape)
    
    # Safety check: one feature row per rerank pair
    assert X.shape[0] == len(df), (
        f"Feature rows {X.shape[0]} != checkpoint rows {len(df)}"
    )
    
    xgb_prob = xgb_model.predict_proba(X)[:, 1]
    
    result = df.copy()
    result["xgb_probability"] = xgb_prob
    
    result["borderline"] = (
        (result["xgb_probability"] >= XGB_THRESHOLD) &
        (result["xgb_probability"] < LGB_THRESHOLD)
    )
    
    result["hybrid_keep"] = (
        (result["xgb_probability"] >= LGB_THRESHOLD)
        |
        (
            (result["xgb_probability"] >= XGB_THRESHOLD) &
            (result["xgb_probability"] < LGB_THRESHOLD) &
            (result["lgb_probability"] >= LGB_THRESHOLD)
        )
    )
    
    result.to_parquet(out_path, index=False)

    # Reconstruct XGB probability
    xgb_prob = xgb_model.predict_proba(X)[:, 1]

    result = df.copy()
    result["xgb_probability"] = xgb_prob

    # Mark the borderline region
    result["borderline"] = (
        (result["xgb_probability"] >= XGB_THRESHOLD) &
        (result["xgb_probability"] < LGB_THRESHOLD)
    )

    # Final hybrid decision
    result["hybrid_keep"] = (
        (result["xgb_probability"] >= LGB_THRESHOLD) |
        (
            (result["xgb_probability"] >= XGB_THRESHOLD) &
            (result["xgb_probability"] < LGB_THRESHOLD) &
            (result["lgb_probability"] >= LGB_THRESHOLD)
        )
    )

    result.to_parquet(out_path, index=False)

print("\nDone.")

[1/89] chunk_00000.parquet rows=48,346


D:\V-Max_Hackerz\.venv\Lib\site-packages\pandas\core\base.py:689: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)


Feature rows: 48346
Feature columns: 17
Feature matrix: (48346, 17)
[2/89] chunk_00001.parquet rows=48,247


D:\V-Max_Hackerz\.venv\Lib\site-packages\pandas\core\base.py:689: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)


Feature rows: 48247
Feature columns: 17
Feature matrix: (48247, 17)
[3/89] chunk_00002.parquet rows=48,338


...

Feature rows: 48334
Feature columns: 17
Feature matrix: (48334, 17)
[88/89] chunk_00087.parquet rows=48,313


D:\V-Max_Hackerz\.venv\Lib\site-packages\pandas\core\base.py:689: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)


Feature rows: 48313
Feature columns: 17
Feature matrix: (48313, 17)
[89/89] chunk_00088.parquet rows=9,118


D:\V-Max_Hackerz\.venv\Lib\site-packages\pandas\core\base.py:689: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values, dtype=dtype)


Feature rows: 9118
Feature columns: 17
Feature matrix: (9118, 17)

Done.


In [20]:
all_parts = []

for path in sorted(OUT_DIR.glob("chunk_*.parquet")):
    all_parts.append(pd.read_parquet(path))

rerank = pd.concat(all_parts, ignore_index=True)

print("Total rerank pairs:", len(rerank))
print("Expected approximately:", 4_409_432)

print("\nXGB probability:")
print(rerank["xgb_probability"].describe())

print("\nBorderline pairs:", rerank["borderline"].sum())
print("Hybrid retained:", rerank["hybrid_keep"].sum())
print("Hybrid removed:", (~rerank["hybrid_keep"]).sum())

print("\nLGB probability:")
print(rerank["lgb_probability"].describe())

Total rerank pairs: 4262476
Expected approximately: 4409432

XGB probability:
count    4.262476e+06
mean     7.392993e-05
std      1.021460e-06
min      7.388604e-05
25%      7.388604e-05
50%      7.388604e-05
75%      7.388604e-05
max      9.769658e-05
Name: xgb_probability, dtype: float64

Borderline pairs: 0
Hybrid retained: 0
Hybrid removed: 4262476

LGB probability:
count    4.262476e+06
mean     9.608195e-01
std      6.660611e-02
min      6.800001e-01
25%      9.553662e-01
50%      9.936277e-01
75%      9.993253e-01
max      9.999983e-01
Name: lgb_probability, dtype: float64


In [21]:
import numpy as np

print("X shape:", X.shape)
print("X dtype:", X.dtype)

print("\nNaN count:", np.isnan(X).sum())
print("Inf count:", np.isinf(X).sum())

print("\nFeature min:")
print(np.nanmin(X, axis=0))

print("\nFeature max:")
print(np.nanmax(X, axis=0))

print("\nFeature mean:")
print(np.nanmean(X, axis=0))

print("\nXGB model:")
print("n_features_in_:", getattr(xgb_model, "n_features_in_", "N/A"))

test_prob = xgb_model.predict_proba(X[:20])[:, 1]

print("\nFirst 20 XGB probabilities:")
print(test_prob)

print("\nUnique probabilities:", len(np.unique(test_prob)))

X shape: (9118, 17)
X dtype: float32

NaN count: 0
Inf count: 0

Feature min:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Feature max:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]

Feature mean:
[0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.0016451]

XGB model:
n_features_in_: 17

First 20 XGB probabilities:
[7.388604e-05 7.388604e-05 7.388604e-05 7.388604e-05 7.388604e-05
 7.388604e-05 7.388604e-05 7.388604e-05 7.388604e-05 7.388604e-05
 7.388604e-05 7.388604e-05 7.388604e-05 7.388604e-05 7.388604e-05
 7.388604e-05 7.388604e-05 7.388604e-05 7.388604e-05 7.388604e-05]

Unique probabilities: 1


In [22]:
import inspect

print("Signature:")
print(inspect.signature(build_chunk_features_fast))

print("\nSource:")
print(inspect.getsource(build_chunk_features_fast))

Signature:
(candidate_pairs_chunk)

Source:
def build_chunk_features_fast(candidate_pairs_chunk):
    """
    Build the exact same 17 features used during training,
    but calculate the expensive fuzzy features in bulk.
    """

    # --------------------------------------------------
    # 1. Flatten candidate_pairs into individual pairs
    # --------------------------------------------------

    s1_ids = []
    candidate_ids = []

    for row in candidate_pairs_chunk.itertuples(index=False):

        s1_id = row.source1_entity_id
        candidate_string = row.candidate_entity_ids

        if pd.isna(candidate_string) or not candidate_string:
            continue

        ids = [
            x.strip()
            for x in candidate_string.split(",")
            if x.strip()
        ]

        for cid in ids:
            s1_ids.append(s1_id)
            candidate_ids.append(cid)

    if not candidate_ids:
        return (
            np.empty((0, 17), dtype=np.float32),
           

In [23]:
print("S1 cache:")
print("shape:", s1_cache.shape)
print("columns:", s1_cache.columns.tolist())
print("index sample:", s1_cache.index[:5].tolist())

print("\nCandidate cache:")
print("shape:", candidate_cache.shape)
print("columns:", candidate_cache.columns.tolist())
print("index sample:", candidate_cache.index[:5].tolist())

print("\nCurrent feature input:")
print(feature_input.head())

print("\nFirst S1 IDs:")
print(feature_input["source1_entity_id"].head().tolist())

print("\nFirst candidate IDs:")
print(
    feature_input["candidate_entity_ids"]
    .iloc[0]
    .split(",")[:5]
)

S1 cache:
shape: (1732544, 9)
columns: ['entity_id', 'business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'country_clean', 'name_missing', 'address_missing']
index sample: [0, 1, 2, 3, 4]

Candidate cache:
shape: (9969589, 8)
columns: ['business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'country_clean', 'name_missing', 'address_missing']
index sample: ['S2-192345572', 'S2-566025912', 'S2-158121477', 'S2-89663826', 'S2-884102769']

Current feature input:
  source1_entity_id candidate_entity_ids
0      S1-486651278         S3-667326708
1      S1-738380571         S2-177917358
2      S1-738380571         S2-620700128
3      S1-738380571         S3-609933190
4      S1-738380571         S3-686355695

First S1 IDs:
['S1-486651278', 'S1-738380571', 'S1-738380571', 'S1-738380571', 'S1-738380571']

First candidate IDs:
['S3-667326708']


In [24]:
# Fix S1 cache index
s1_cache = s1_cache.set_index("entity_id", drop=False)

print("S1 cache index sample:")
print(s1_cache.index[:5].tolist())

# Verify that actual S1 IDs can now be found
test_ids = feature_input["source1_entity_id"].head().tolist()

print("\nTest S1 IDs:")
print(test_ids)

print("\nFound in cache:")
print(s1_cache.index.isin(test_ids).tolist())

S1 cache index sample:
['S1-714132312', 'S1-106407869', 'S1-156285671', 'S1-689823050', 'S1-921369899']

Test S1 IDs:
['S1-486651278', 'S1-738380571', 'S1-738380571', 'S1-738380571', 'S1-738380571']

Found in cache:


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [25]:
test_ids = feature_input["source1_entity_id"].head().tolist()

found = s1_cache.index.isin(test_ids)

print("Test IDs:", test_ids)
print("Found count:", found.sum(), "/", len(test_ids))
print("All found:", found.all())

Test IDs: ['S1-486651278', 'S1-738380571', 'S1-738380571', 'S1-738380571', 'S1-738380571']
Found count: 2 / 5
All found: False


In [26]:
print("Available cache variables:")
print("s1_cache:", s1_cache.shape)

# Check whether the test cache exists
print("test_s1_cache:", test_s1_cache.shape)
print(test_s1_cache.index[:5].tolist())

Available cache variables:
s1_cache: (1732544, 9)


NameError: name 'test_s1_cache' is not defined

In [27]:
print("s1_cache shape:", s1_cache.shape)
print("Index type:", type(s1_cache.index))
print("Index name:", s1_cache.index.name)

print("\nFirst 10 index values:")
print(s1_cache.index[:10].tolist())

print("\nFirst 10 entity_id values:")
print(s1_cache["entity_id"].head(10).tolist())

s1_cache shape: (1732544, 9)
Index type: <class 'pandas.Index'>
Index name: entity_id

First 10 index values:
['S1-714132312', 'S1-106407869', 'S1-156285671', 'S1-689823050', 'S1-921369899', 'S1-481669221', 'S1-909865979', 'S1-280204013', 'S1-742053041', 'S1-913506265']

First 10 entity_id values:
['S1-714132312', 'S1-106407869', 'S1-156285671', 'S1-689823050', 'S1-921369899', 'S1-481669221', 'S1-909865979', 'S1-280204013', 'S1-742053041', 'S1-913506265']


In [28]:
test_ids = feature_input["source1_entity_id"].drop_duplicates().head(20).tolist()

found = s1_cache.index.isin(test_ids)

print("Test IDs checked:", len(test_ids))
print("Found:", found.sum())
print("Missing:", len(test_ids) - found.sum())

print("\nMissing IDs:")
print([x for x in test_ids if x not in s1_cache.index])

Test IDs checked: 20
Found: 20
Missing: 0

Missing IDs:
[]


In [29]:
X, s1_ids, candidate_ids = build_chunk_features_fast(
    feature_input
)

print("X shape:", X.shape)
print("Expected rows:", len(candidate_ids))
print("NaN count:", np.isnan(X).sum())
print("Inf count:", np.isinf(X).sum())

X shape: (9118, 17)
Expected rows: 9118
NaN count: 0
Inf count: 0


In [30]:
test_prob = xgb_model.predict_proba(X[:100])[:, 1]

print("First 20 XGB probabilities:")
print(test_prob[:20])

print("Unique probabilities:", len(np.unique(test_prob)))
print("Min:", test_prob.min())
print("Max:", test_prob.max())

First 20 XGB probabilities:
[0.9988625  0.99630153 0.84128785 0.9713414  0.8492732  0.9979473
 0.99814165 0.99826294 0.8896372  0.99715084 0.99624866 0.7400328
 0.9864871  0.99828917 0.99830365 0.99907243 0.8795255  0.96837664
 0.99924845 0.9967649 ]
Unique probabilities: 100
Min: 0.6785378
Max: 0.99974936


In [31]:
# Use the corrected TEST S1 cache
print("S1 cache:", s1_cache.shape)
print("Candidate cache:", candidate_cache.shape)

# Load first LightGBM rerank checkpoint
rerank_file = sorted(
    (RESULTS_DIR / "lgb_rerank_checkpoints").glob("chunk_*.parquet")
)[0]

df = pd.read_parquet(rerank_file)

print("Rerank checkpoint:", rerank_file.name)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print(df.head())

S1 cache: (1732544, 9)
Candidate cache: (9969589, 8)
Rerank checkpoint: chunk_00000.parquet
Rows: 48346
Columns: ['source1_entity_id', 'candidate_entity_id', 'lgb_probability']
  source1_entity_id candidate_entity_id  lgb_probability
0      S1-714132312        S3-625880872         0.998688
1      S1-714132312        S3-867809779         0.976911
2      S1-106407869        S2-705547832         0.960914
3      S1-106407869        S3-613056593         0.973223
4      S1-156285671        S2-172049742         0.999205


In [32]:
# Prepare the checkpoint for the feature builder
feature_input = df.rename(
    columns={
        "candidate_entity_id": "candidate_entity_ids"
    }
)[
    ["source1_entity_id", "candidate_entity_ids"]
].copy()

print("Feature input rows:", len(feature_input))

# Build the exact 17 XGB features
X, s1_ids, candidate_ids = build_chunk_features_fast(
    feature_input
)

print("Feature shape:", X.shape)
print("NaN:", np.isnan(X).sum())
print("Inf:", np.isinf(X).sum())

# XGBoost probability
xgb_prob = xgb_model.predict_proba(X)[:, 1]

print("\nXGB probability statistics:")
print(pd.Series(xgb_prob).describe())

print("\nUnique XGB probabilities:", len(np.unique(xgb_prob)))

Feature input rows: 48346
Feature shape: (48346, 17)
NaN: 0
Inf: 0

XGB probability statistics:
count    48346.000000
mean         0.945991
std          0.079472
min          0.463305
25%          0.926923
50%          0.986531
75%          0.997557
max          0.999877
dtype: float64

Unique XGB probabilities: 40960


In [33]:
rerank_test = df.copy()

rerank_test["xgb_probability"] = xgb_prob

print(rerank_test.head())

  source1_entity_id candidate_entity_id  lgb_probability  xgb_probability
0      S1-714132312        S3-625880872         0.998688         0.993439
1      S1-714132312        S3-867809779         0.976911         0.985336
2      S1-106407869        S2-705547832         0.960914         0.968316
3      S1-106407869        S3-613056593         0.973223         0.968372
4      S1-156285671        S2-172049742         0.999205         0.994649


In [34]:
XGB_THRESHOLD = 0.67
LGB_THRESHOLD = 0.73

print("XGB >= 0.67:", 
      (rerank_test["xgb_probability"] >= XGB_THRESHOLD).sum())

print("XGB >= 0.73:", 
      (rerank_test["xgb_probability"] >= LGB_THRESHOLD).sum())

print("LGB >= 0.73:", 
      (rerank_test["lgb_probability"] >= LGB_THRESHOLD).sum())

print("\nBoth >= thresholds:",
      (
          (rerank_test["xgb_probability"] >= XGB_THRESHOLD) &
          (rerank_test["lgb_probability"] >= LGB_THRESHOLD)
      ).sum())

XGB >= 0.67: 48300
XGB >= 0.73: 46682
LGB >= 0.73: 47468

Both >= thresholds: 47436


In [35]:
print("XGB only:",
      (
          (rerank_test["xgb_probability"] >= XGB_THRESHOLD) &
          (rerank_test["lgb_probability"] < LGB_THRESHOLD)
      ).sum())

print("LGB only:",
      (
          (rerank_test["xgb_probability"] < XGB_THRESHOLD) &
          (rerank_test["lgb_probability"] >= LGB_THRESHOLD)
      ).sum())

print("Both:",
      (
          (rerank_test["xgb_probability"] >= XGB_THRESHOLD) &
          (rerank_test["lgb_probability"] >= LGB_THRESHOLD)
      ).sum())

print("Neither:",
      (
          (rerank_test["xgb_probability"] < XGB_THRESHOLD) &
          (rerank_test["lgb_probability"] < LGB_THRESHOLD)
      ).sum())

XGB only: 864
LGB only: 32
Both: 47436
Neither: 14


NameError: name 'xgb_val_prob' is not defined